# 03 Labels: Wann ist eine Station auffällig?

Teil 2: Begehungen und Wirkung

**Worum es geht**
- Kein Modell, sondern die Frage davor: Was soll ein Modell überhaupt erkennen?
- Vorschlag: eine Definition von „auffällig“ allein aus den Messdaten, je Station und Monat. Danach eine ehrliche Prüfung, ob sie hält.
- Grund: Cluster taugen nicht als Label. Das zeigt Teil 3 vorab ([Notebook 07](../3_clustering/07_stationscluster.ipynb): kein natürliches Gruppenmuster, Silhouette unter 0,25). Die Änderungs-Kategorie misst, wie viel der Monteur verstellt hat, nicht wie schlecht die Station lief ([Notebook 02](02_wirkung.ipynb)). `n_actions > 0` ist bei 76 % der Stationen wahr und trennt nichts.
- Idee: Eine Hausübergabestation (HAST) soll dem heißen Wasser Wärme entziehen und es kalt zurückschicken. Fließt viel Wasser, kommt es aber fast genauso heiß zurück, wurde keine Wärme übertragen. Das ist messbar und braucht keine Begehung.

**Daten**
- liest: stündliche Zeitreihen aus `data/processed/real_meters/`, Begehungen über `src/load_begehungen.py`, Kodierbogen `data/processed/kodierbogen_freitexte.xlsx` (optional, Abschnitt 7)
- schreibt: `data/processed/station_monat_label.csv` mit Lauf-Kontext `station_monat_label.json`
- vorher ausführen: Notebook 00

**Code**
- Begehungs-Loader in `src/load_begehungen.py`, Tests in `tests/test_load_begehungen.py`

## 1 Setup und Daten laden

- Bibliotheken laden, Projektwurzel finden, Zeitreihen auflisten.

In [1]:
import os as _os
from pathlib import Path as _Path
#Arbeitsordner ist immer notebooks/, auch aus den Teil-Ordnern
_os.chdir(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "src").is_dir()) / "notebooks")
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", message=".*Data Validation extension.*")

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load_begehungen import begehungen_mit_zaehler

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.facecolor"] = "white"

METER_DIR = ROOT / "data" / "processed" / "real_meters"
DATEIEN = sorted(METER_DIR.glob("*.csv"))
print(f"{len(DATEIEN)} Zaehler-Zeitreihen gefunden")

93 Zaehler-Zeitreihen gefunden


- Das Notebook sucht die Projektwurzel selbst (nach oben, bis `src/` auftaucht) und läuft deshalb aus `notebooks/` und aus dem Hauptordner.
- Der Begehungs-Loader aus `src/load_begehungen.py` (entstanden in [Notebook 02](02_wirkung.ipynb)) wird erst in Abschnitt 6 gebraucht.
- `DATEIEN`: 93 stündliche Zeitreihen je Zähler aus [Notebook 00](../1_daten/00_daten.ipynb), dort aus überlappenden Exporten zusammengeführt.
- Die Zeile mit `warnings` unterdrückt nur eine harmlose Meldung beim Excel-Lesen.

## 2 Monatswerte je Station

- Aus den Stundenwerten wird je Station und Monat ein typischer Wert (Median) für Durchfluss, Spreizung, Rücklauf und Leistung. Eine Zeile ist eine Station in einem Monat.
- Durchfluss = Wasser durch die Station in l/h. Spreizung = Vorlauf minus Rücklauf in Kelvin. Rücklauf = Temperatur des Wassers, das zurück ins Netz geht.
- Median statt Mittelwert, damit einzelne Ausreißer wie eine kurze heiße Zapfung den Wert nicht verziehen.

In [2]:
MIN_STUNDEN = 400

SPALTEN = {
    "Volume flow (l/h)": "fluss",
    "Temperature difference (°C)": "spreizung",
    "Return temperature (°C)": "ruecklauf",
    "Power (kW)": "leistung",
}


def monatswerte(pfad):
    """Monatliche Mittelwerte (Median) einer Zaehler-Zeitreihe."""
    df = pd.read_csv(pfad, parse_dates=["Timestamp"]).set_index("Timestamp").sort_index()
    df = df.rename(columns=SPALTEN)
    monate = df.resample("MS").agg(
        fluss=("fluss", "median"),
        spreizung=("spreizung", "median"),
        ruecklauf=("ruecklauf", "median"),
        leistung=("leistung", "median"),
        stunden=("fluss", "count"),
    )
    monate = monate[monate["stunden"] >= MIN_STUNDEN].copy()
    if monate.empty:
        return None
    monate["zaehler"] = int(pfad.stem)
    return monate.reset_index().rename(columns={"Timestamp": "monat"})


teile = [monatswerte(p) for p in DATEIEN]
monate = pd.concat([t for t in teile if t is not None], ignore_index=True)

print(f"Station-Monate mit genug Messdaten: {len(monate)}")
print(f"Stationen                         : {monate['zaehler'].nunique()}")
print(f"Zeitraum                          : {monate['monat'].min():%Y-%m} bis {monate['monat'].max():%Y-%m}")
display(monate.head(3))

Station-Monate mit genug Messdaten: 2300
Stationen                         : 93
Zeitraum                          : 2022-11 bis 2026-07


,monat,fluss,spreizung,ruecklauf,leistung,stunden,zaehler
0,2022-11-01,80.0,18.9,53.9,2.030,444,53198226
1,2022-12-01,190.0,25.4,50.4,6.495,744,53198226
2,2023-01-01,170.0,27.5,48.0,6.090,743,53198226


- 2300 Station-Monate mit genug Messdaten, 93 Stationen, November 2022 bis Juli 2026.
- `MIN_STUNDEN = 400`: Monate mit weniger Messstunden fallen weg. Die Abdeckung schwankt je Station zwischen 46 % und 100 % ([Notebook 00](../1_daten/00_daten.ipynb)); ein Monatswert aus 30 Stunden wäre wertlos.
- Die Spalten werden über ein Mapping umbenannt, nicht über die Position. Mit Positionen werden leicht Spreizung und Leistung vertauscht. Dieser Fehler ist beim Vorbereiten passiert und hat das Ergebnis vervierfacht.

## 3 Das Label festlegen

- Regel: Durchfluss über 50 l/h und gleichzeitig Spreizung unter 8 K = auffällig.
- Anschaulich: Es fließt viel Wasser, es kommt aber fast so heiß zurück, wie es ankam. Die Station wälzt um, ohne Wärme abzugeben.
- Urteil je Station und Monat, weil eine Station jahrelang auffällig und danach in Ordnung sein kann.

In [3]:
FLUSS_SCHWELLE = 50   #l/h
SPREIZUNG_SCHWELLE = 8  #Kelvin

monate["auffaellig"] = (
    (monate["fluss"] > FLUSS_SCHWELLE) & (monate["spreizung"] < SPREIZUNG_SCHWELLE)
)

betroffen = monate.loc[monate["auffaellig"], "zaehler"].nunique()
print(f"Regel: Durchfluss ueber {FLUSS_SCHWELLE} l/h UND Spreizung unter {SPREIZUNG_SCHWELLE} K\n")
print(f"auffaellige Station-Monate : {int(monate['auffaellig'].sum())} von {len(monate)}"
      f" = {monate['auffaellig'].mean():.1%}")
print(f"betroffene Stationen       : {betroffen} von {monate['zaehler'].nunique()}")

BEISPIEL = 68956351
bsp = monate[monate["zaehler"] == BEISPIEL].set_index("monat")
print(f"\nGegenprobe Zaehler {BEISPIEL}, Reparatur war im September 2024:")
display(bsp.loc["2024-06":"2024-12", ["fluss", "spreizung", "auffaellig"]].round(1))

Regel: Durchfluss ueber 50 l/h UND Spreizung unter 8 K

auffaellige Station-Monate : 266 von 2300 = 11.6%
betroffene Stationen       : 24 von 93

Gegenprobe Zaehler 68956351, Reparatur war im September 2024:


,fluss,spreizung,auffaellig
monat,,,
2024-06-01,671.0,0.1,True
2024-07-01,507.0,0.1,True
2024-08-01,494.0,0.1,True
2024-09-01,3.0,0.4,False
2024-10-01,3.0,18.0,False
2024-11-01,44.5,32.8,False
2024-12-01,73.0,35.5,False


- 266 von 2300 Station-Monaten sind auffällig (11,6 %), 24 von 93 Stationen betroffen.
- Gegenprobe: Bei Zähler 68956351 wurde am 09.09.2024 ein Ventil geschlossen, der Durchfluss fiel über Nacht von 499 auf 0 l/h ([Notebook 02](02_wirkung.ipynb)).
- Die Regel kennt dieses Datum nicht, kippt aber genau im September 2024 von auffällig auf unauffällig. Die Spreizung steigt in den Folgemonaten von rund 0 auf über 30 K. Die Regel bildet diesen Fall also richtig ab.
- Die Schwellen 50 l/h und 8 K sind gesetzt, nicht gemessen. Abschnitt 4 prüft, ob sie tragen.

## 4 Sind die Schwellen robust?

- Eine Definition, die bei leicht anderen Zahlen ganz andere Ergebnisse liefert, wäre willkürlich.
- Deshalb neun Schwellenpaare: Durchfluss 40, 50, 60 l/h mal Spreizung 6, 8, 10 K.
- Gezählt wird, wie viele Monate und welche Stationen getroffen werden.

In [4]:
fluss_werte = [40, 50, 60]
spreizung_werte = [6, 8, 10]

raster = []
mengen = {}
for f in fluss_werte:
    for s in spreizung_werte:
        treffer = (monate["fluss"] > f) & (monate["spreizung"] < s)
        stationen = set(monate.loc[treffer, "zaehler"])
        mengen[(f, s)] = stationen
        raster.append({"Durchfluss >": f, "Spreizung <": s,
                       "Monate": int(treffer.sum()), "Stationen": len(stationen)})

print("Wie viele Treffer ergeben andere Schwellen?")
display(pd.DataFrame(raster).pivot(index="Durchfluss >", columns="Spreizung <",
                                   values=["Monate", "Stationen"]))

basis = mengen[(FLUSS_SCHWELLE, SPREIZUNG_SCHWELLE)]
streng = mengen[(60, 6)]
locker = mengen[(40, 10)]
print(f"\nStationen bei unserer Regel (50/8) : {len(basis)}")
print(f"strengste Variante (60/6)          : {len(streng)}, davon in unserer Menge: {len(streng & basis)}")
print(f"lockerste Variante (40/10)         : {len(locker)}, davon in unserer Menge: {len(locker & basis)}")

Wie viele Treffer ergeben andere Schwellen?


Monate           Stationen        
Spreizung <      6    8    10        6   8   10
Durchfluss >                                   
40              241  273  322        22  25  30
50              237  266  307        22  24  29
60              234  263  300        21  24  26


Stationen bei unserer Regel (50/8) : 24
strengste Variante (60/6)          : 21, davon in unserer Menge: 21
lockerste Variante (40/10)         : 30, davon in unserer Menge: 24


- Auffällige Monate je Variante zwischen 234 und 322, Stationen zwischen 21 und 30.
- Die 21 Stationen der strengsten Variante (60/6) stecken alle in unserer Menge (50/8, 24 Stationen). Die lockerste Variante (40/10) hat 30 Stationen und enthält alle 24.
- Es gibt also einen stabilen Kern, der unabhängig von der genauen Schwelle auffällt. Die Definition hängt nicht an einer glücklich gewählten Zahl.
- Anders die Baseline in `src/fault_detection.py`: Ihre Schwellen wurden auf synthetischen Daten kalibriert und schlagen bei fast allen echten Zählern an.

## 5 Wann wechselt der Zustand?

- Ein brauchbares Label sollte sich über die Zeit sinnvoll verhalten.
- Gezählt wird, wie oft eine Station von unauffällig auf auffällig kippt (Beginn) und zurück (Ende).
- Dazu zwei triviale Vergleichsmodelle: „wie im Vormonat“ und „immer unauffällig“.

In [5]:
monate = monate.sort_values(["zaehler", "monat"]).reset_index(drop=True)
monate["vormonat"] = monate.groupby("zaehler")["auffaellig"].shift()

beginn = monate[(monate["auffaellig"]) & (monate["vormonat"] == False)]
ende = monate[(~monate["auffaellig"]) & (monate["vormonat"] == True)]

print(f"Beginn eines auffaelligen Zustands: {len(beginn)} mal, bei {beginn['zaehler'].nunique()} Stationen")
print(f"Ende eines auffaelligen Zustands  : {len(ende)} mal, bei {ende['zaehler'].nunique()} Stationen")

#schon im ersten Messmonat auffällig
erste = monate.groupby("zaehler").first()
print(f"schon im ersten Messmonat auffaellig: {int(erste['auffaellig'].sum())} Stationen"
      f" - deren Beginn wurde nie beobachtet")

vergleich = monate.dropna(subset=["vormonat"])
print(f"\nTriviales Modell 'wie im Vormonat': {(vergleich['auffaellig'] == vergleich['vormonat']).mean():.1%} richtig")
print(f"Triviales Modell 'immer unauffaellig': {(~vergleich['auffaellig']).mean():.1%} richtig")

Beginn eines auffaelligen Zustands: 36 mal, bei 21 Stationen
Ende eines auffaelligen Zustands  : 37 mal, bei 22 Stationen
schon im ersten Messmonat auffaellig: 7 Stationen - deren Beginn wurde nie beobachtet

Triviales Modell 'wie im Vormonat': 96.7% richtig
Triviales Modell 'immer unauffaellig': 88.3% richtig


- Beginn 36-mal bei 21 Stationen, Ende 37-mal bei 22 Stationen. 7 Stationen waren schon im ersten Messmonat auffällig; ihr Beginn ist nie beobachtet.
- Der Zustand ändert sich selten: „wie im Vormonat“ liegt zu 96,7 % richtig, „immer unauffällig“ zu 88,3 %, ohne etwas verstanden zu haben.
- Derselbe Fallstrick wie bei der Baseline auf synthetischen Daten, die 100 % erreichte und trotzdem nichts aussagte.
- Folge: Interessant sind nur die wenigen Wechsel. Ein späteres Modell muss auf den Wechseln bewertet werden, nicht auf allen Zeilen.

## 6 Stimmt das Label mit den Begehungen überein?

- Bisher prüft sich die Regel nur an sich selbst. Unabhängige Quelle sind die Notizen der Monteure.
- Frage 1: Ist eine Station häufiger auffällig, wenn ihr Freitext ein hydraulisches Thema nennt? Stichwortsuche nach Ventil, Zirkulation, Schmutzfänger, Filter, Durchfluss, Pumpe, Kurzschluss.
- Frage 2: Fallen Zustandswechsel in einen Monat mit Begehung?

In [6]:
VERFUEGBAR = {int(p.stem) for p in DATEIEN}
beg = begehungen_mit_zaehler(available=VERFUEGBAR)

#irgendwann auffällig?
je_station = monate.groupby("zaehler")["auffaellig"].any()
beg["physik_auffaellig"] = beg["meter"].map(je_station).fillna(False)

MUSTER = r"ventil|zirk|schmutzf|filter|durchfluss|pumpe|kurzschluss"
beg["text_hydraulik"] = beg["sonstiges"].fillna("").str.contains(MUSTER, case=False, regex=True)

kreuz = pd.crosstab(beg["text_hydraulik"], beg["physik_auffaellig"])
print("Freitext nennt Hydraulik (Zeilen) vs. Label auffaellig (Spalten):")
display(kreuz)

mit = beg.loc[beg["text_hydraulik"], "physik_auffaellig"].mean()
ohne = beg.loc[~beg["text_hydraulik"], "physik_auffaellig"].mean()
print(f"Freitext nennt Hydraulik  : {mit:.0%} auffaellig (n={int(beg['text_hydraulik'].sum())})")
print(f"Freitext nennt es nicht   : {ohne:.0%} auffaellig (n={int((~beg['text_hydraulik']).sum())})")
if kreuz.shape == (2, 2):
    print(f"Fisher-Test: p = {stats.fisher_exact(kreuz.values)[1]:.3f}")

bd = beg.dropna(subset=["datum", "meter"]).copy()
begehungsmonate = set(zip(bd["meter"].astype(int), bd["datum"].dt.to_period("M").astype(str)))
for name, tab in [("Beginn", beginn), ("Ende", ende)]:
    paare = zip(tab["zaehler"].astype(int), tab["monat"].dt.to_period("M").astype(str))
    treffer = sum(p in begehungsmonate for p in paare)
    print(f"{name}: {treffer} von {len(tab)} Wechseln fallen in einen Begehungsmonat")

Freitext nennt Hydraulik (Zeilen) vs. Label auffaellig (Spalten):


physik_auffaellig,False,True
text_hydraulik,,
False,37,8
True,15,6


Freitext nennt Hydraulik  : 29% auffaellig (n=21)
Freitext nennt es nicht   : 18% auffaellig (n=45)
Fisher-Test: p = 0.346


Beginn: 2 von 36 Wechseln fallen in einen Begehungsmonat
Ende: 2 von 37 Wechseln fallen in einen Begehungsmonat


- Frage 1: Mit Hydraulik-Stichwort sind 29 % auffällig (n = 21), ohne 18 % (n = 45). Fisher-Test (prüft bei kleinen Vierfeldertafeln, ob der Unterschied Zufall sein kann) p = 0,346. Die Notizen bestätigen das Label hier nicht. Drei Gründe, die sich nicht trennen lassen:
  - Die Stichwortsuche ist grob; von Hand ließe sich sauberer einteilen.
  - Die 66 Begehungen decken nur einen Teil der 93 Stationen ab, und die Regel trifft davon wenige.
  - Notiz und Regel beschreiben vielleicht verschiedene Dinge.
- Frage 2: Nur 2 von 36 Beginnen und 2 von 37 Enden fallen in einen Begehungsmonat. Diese wenigen Fälle passen zu [Notebook 02](02_wirkung.ipynb):
  - Die beiden Enden gehören zu den Stationen mit der stärksten gemessenen Verbesserung, darunter der Ventilfall.
  - Die beiden Beginne sind die Stationen, deren Rücklauf nach der Begehung deutlich stieg: 72167783 von 18,6 auf 71,0 °C, 80912173 von 54,0 auf 67,3 °C.
- Die Stationen ändern ihren Zustand also überwiegend ohne erkennbaren äußeren Anlass. Das passt dazu, dass Notebook 02 keinen statistisch gesicherten Effekt der Begehungen fand.
- Die Stichwortsuche ist grob. Abschnitt 7 wiederholt Frage 1 mit einer Einordnung von Hand.

## 7 Abgleich mit der Handkodierung

- Alle 57 Freitextnotizen wurden zusätzlich von Hand eingeordnet, und zwar blind: ohne zu sehen, welche Station die Regel als auffällig einstuft.
- Spalte `passt_zur_regel` (ja, nein, unklar) und eine Befundart; Verbindung zum Label aus Abschnitt 3 über die Zählernummer.
- Vergleich insgesamt, bei schweren Fällen (mindestens 10 auffällige Monate) und je Befundart. Fehlt der Kodierbogen, wird der Abschnitt übersprungen.

In [7]:
KODIERBOGEN = ROOT / "data" / "processed" / "kodierbogen_freitexte.xlsx"

if not KODIERBOGEN.is_file():
    print(f"{KODIERBOGEN.name} nicht gefunden - Abschnitt uebersprungen.")
else:
    kod = pd.read_excel(KODIERBOGEN, sheet_name="Kodierung")
    je_station = monate.groupby("zaehler")["auffaellig"].any()
    auff_monate = monate.groupby("zaehler")["auffaellig"].sum()
    kod["label"] = kod["Zaehler"].map(je_station)
    kod["auff_monate"] = kod["Zaehler"].map(auff_monate)

    print(f"kodierte Notizen: {len(kod)}")
    display(kod["passt_zur_regel"].value_counts().to_frame("n"))

    bewertbar = kod.dropna(subset=["label"])
    print("Anteil der Stationen, die die Regel auffaellig findet:")
    display(bewertbar.groupby("passt_zur_regel")["label"]
            .agg(Anteil="mean", n="size").round(2))

    jn = bewertbar[bewertbar["passt_zur_regel"].isin(["ja", "nein"])]
    kreuz = pd.crosstab(jn["passt_zur_regel"], jn["label"])
    if kreuz.shape == (2, 2):
        print(f"Fisher-Test ja gegen nein: p = {stats.fisher_exact(kreuz.values)[1]:.3f}")

    schwer = bewertbar[bewertbar["auff_monate"] >= 10]
    print(f"\nStationen mit mindestens 10 auffaelligen Monaten: {schwer['Zaehler'].nunique()}")
    display(schwer[["Zaehler", "auff_monate", "passt_zur_regel", "befundart"]]
            .sort_values("auff_monate", ascending=False))

    n_ja = int((kod["passt_zur_regel"] == "ja").sum())
    verpasst = kod[(kod["passt_zur_regel"] == "ja") & (kod["auff_monate"] == 0)]
    print(f"als ja kodiert, aber kein einziger auffaelliger Monat: {len(verpasst)} von {n_ja}")

    print("\nLabel-Anteil je Befundart:")
    display(bewertbar.groupby("befundart")["label"]
            .agg(Anteil="mean", n="size").round(2).sort_values("Anteil", ascending=False))

    print(f"\nZeilen ohne Labeldaten: {int(kod['label'].isna().sum())}")

kodierbogen_freitexte.xlsx nicht gefunden - Abschnitt uebersprungen.


- Insgesamt stimmt nur die Richtung: Bei „ja“ findet die Regel 33 % auffällig (n = 12), bei „nein“ 8 % (n = 12), Fisher-Test p = 0,317. Bei 14 ja- und 13 nein-Notizen ist mehr kaum zu erwarten. 30 der 57 Notizen sind „unklar“ und sagen zur Frage nichts.
- Die Regel übersieht viele dokumentierte Mängel: 8 von 14 mit „ja“ kodierten Stationen haben keinen einzigen auffälligen Monat, darunter ein defekter Stellmotor, dessen Ventil aufgedrückt wurde. Naheliegende Gründe:
  - zeitweise Fehler verschwinden im Monatsmedian (eine Notiz nennt ein Ventil, das nur „hin und wieder“ hängt)
  - Fehler auf der Hausseite (Heizkreispumpe, Fußbodenheizung ohne Mischer) zeigen sich auf der Netzseite nicht zwangsläufig als Dauerdurchfluss
- Bei schweren Fällen passt es gut: Von 6 Stationen mit mindestens 10 auffälligen Monaten ist keine mit „nein“ kodiert. Von den 4 mit über 20 auffälligen Monaten sind 3 „ja“, eine (80912182) „unklar“. Zirkulation (67 %) und Ventil/Hydraulik (43 %) treffen häufiger auf auffällige Stationen als Regelung und Konfiguration (je 0 %).
- Folge: Die Regel ist kein allgemeiner Defektdetektor, sondern ein Detektor für schwere, monatelang anhaltende Zustände. In dieser engeren Rolle ist sie belastbar und sollte so benannt werden.

## 8 Ergebnis speichern

- Schreibt die Label-Tabelle und den Lauf-Kontext nach `data/processed/`.

In [8]:
ausgabe = ROOT / "data" / "processed" / "station_monat_label.csv"
spalten = ["zaehler", "monat", "stunden", "fluss", "spreizung", "ruecklauf",
           "leistung", "auffaellig"]
monate[spalten].to_csv(ausgabe, index=False)

meta = {
    "erzeugt": pd.Timestamp.now().isoformat(timespec="seconds"),
    "regel": f"fluss > {FLUSS_SCHWELLE} l/h UND spreizung < {SPREIZUNG_SCHWELLE} K",
    "min_stunden_je_monat": MIN_STUNDEN,
    "station_monate": int(len(monate)),
    "stationen": int(monate["zaehler"].nunique()),
    "auffaellige_monate": int(monate["auffaellig"].sum()),
    "betroffene_stationen": int(betroffen),
    "wechsel_beginn": int(len(beginn)),
    "wechsel_ende": int(len(ende)),
    "schwellen_geprueft": {"fluss": fluss_werte, "spreizung": spreizung_werte},
}
ausgabe.with_suffix(".json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Label-Tabelle -> {ausgabe.name}  ({len(monate)} Zeilen)")
print(f"Lauf-Kontext  -> {ausgabe.with_suffix('.json').name}")

Label-Tabelle -> station_monat_label.csv  (2300 Zeilen)
Lauf-Kontext  -> station_monat_label.json


- `station_monat_label.csv`: eine Zeile je Station und Monat mit allen Messwerten und dem Urteil, 2300 Zeilen.
- `station_monat_label.json` hält Regel, Schwellen und Kennzahlen fest. So bleibt später nachvollziehbar, unter welchen Annahmen die Tabelle entstand.
- Die Tabelle enthält Zählernummern. `data/processed/` ist über `.gitignore` ausgeschlossen und bleibt es.

## Zusammenfassung & Bewertung

**Ergebnis**
- Aus 66 Begehungszeilen ohne brauchbares Zustands-Label wurden 2300 bewertete Station-Monate. Die Definition stammt nur aus den Messdaten: Durchfluss über 50 l/h und Spreizung unter 8 K.
- 266 Station-Monate (11,6 %) an 24 von 93 Stationen sind auffällig.
- Bausteine: Stundenreihen und Abdeckung aus Notebook 00; aus Notebook 07 (Teil 3) die Hauptkomponentenanalyse, die Rücklauf und Spreizung als wichtigste Achse auswies; aus Notebook 02 Loader und Reparaturfall.

| Prüfung | Ergebnis | Bewertung |
|---|---|---|
| Gegenprobe am bekannten Reparaturfall | Label kippt genau im Reparaturmonat | bestanden |
| Robustheit der Schwellen | stabiler Kern über neun Schwellenpaare | bestanden |
| Zeitliches Verhalten | wenige, klar erkennbare Wechsel | bestanden |
| Freitext, Stichwortsuche | kein gesicherter Zusammenhang | nicht bestanden |
| Handkodierung, alle Fälle | Richtung stimmt, nicht abgesichert | teilweise |
| Handkodierung, schwere Fälle | keine einzige mit „nein“ kodiert | bestanden |

**Belastbar**
- Das Label ist ein geprüfter Vorschlag mit klarer Reichweite: ein Detektor für schwere, monatelang anhaltende Zustände.
- Unter den Stationen mit mindestens zehn auffälligen Monaten ist keine, deren Notiz widerspricht.
- Die Schwellen tragen: stabiler Kern über neun Schwellenpaare.

**Schwach**
- Als allgemeiner Defektdetektor taugt die Regel nicht. Sie übersieht zeitweise Fehler und Fehler auf der Hausseite; 8 von 14 mit „ja“ kodierte Stationen haben keinen auffälligen Monat.
- Der Freitext bestätigt nur die Richtung (Fisher p = 0,346 Stichwortsuche, p = 0,317 Handkodierung).
- Das triviale Modell „wie im Vormonat“ erreicht 96,7 %. Eine Bewertung auf allen Zeilen sagt deshalb nichts.

**Offen**
- Reichweite in der Arbeit benennen: Detektor für Dauerzustände, nicht allgemeines Defekt-Label.
- Schwellen in `src/fault_detection.py` nachjustieren. Die bisherige Regel schlägt bei fast allen echten Zählern an; die Label-Tabelle liefert erstmals einen Maßstab aus echten Daten. Die auffälligen Stationen mit Zeitraum als Wartungsliste an die ÜZ geben.
- Bestätigt sich das Label nicht, ist auch das ein Ergebnis: „defekt“ ließe sich aus diesen Daten nicht eindeutig definieren. Das würde erklären, warum das Clustering in Teil 3 ([Notebook 07](../3_clustering/07_stationscluster.ipynb)) und die bisherigen Klassifikationsversuche kein Signal fanden.